In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('/Users/mebenga/Quantic MSBA/6-MSBA Communicating with Data/Communicating with Data Presentation - Project/wfp_food_prices_database_wacaro.csv')

In [3]:
df.head(10)

,adm0_id,adm0_name,adm1_id,adm1_name,mkt_id,mkt_name,cm_id,cm_name,cur_id,cur_name,pt_id,pt_name,um_id,um_name,mp_month,mp_year,mp_price,mp_commoditysource
0,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,1,2002,145.00,NaN
1,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,1,2003,106.00,NaN
2,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,2,2003,107.50,NaN
3,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,3,2003,95.00,NaN
4,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,4,2003,95.00,NaN
5,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,5,2003,90.00,NaN
6,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,6,2003,90.00,NaN
7,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,7,2003,79.60,NaN
8,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,8,2003,85.00,NaN
9,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,9,2003,91.25,NaN


In [7]:
df.describe()

,adm0_id,adm1_id,mkt_id,cm_id,cur_id,pt_id,um_id,mp_month,mp_year,mp_price,mp_commoditysource
count,332829.000000,332829.000000,332829.000000,332829.000000,332829.0,332829.000000,332829.000000,332829.000000,332829.000000,332829.000000,0.0
mean,118.359332,11081.659017,1222.903401,141.859931,0.0,14.866226,14.470689,6.293496,2014.657893,2615.569742,NaN
std,56.506105,22120.528164,927.312121,133.921878,0.0,0.340410,27.031605,3.350376,5.123588,9079.002350,NaN
min,29.000000,0.000000,124.000000,50.000000,0.0,14.000000,5.000000,1.000000,1990.000000,0.000000,NaN
25%,50.000000,1329.000000,544.000000,65.000000,0.0,15.000000,5.000000,3.000000,2012.000000,144.000000,NaN
50%,144.000000,2008.000000,887.000000,73.000000,0.0,15.000000,5.000000,6.000000,2016.000000,245.000000,NaN
75%,181.000000,2240.000000,1976.000000,141.000000,0.0,15.000000,9.000000,9.000000,2019.000000,487.500000,NaN
max,182.000000,112869.000000,3737.000000,812.000000,0.0,15.000000,161.000000,12.000000,2020.000000,474500.000000,NaN


In [4]:
# Import necessary libraries first
import pandas as pd
import numpy as np
import plotly.io as pio
pio.renderers.default = "notebook_connected"


# Define file path
df = pd.read_csv('/Users/mebenga/Quantic MSBA/6-MSBA Communicating with Data/Communicating with Data Presentation - Project/wfp_food_prices_database_wacaro.csv')

# -----------------------------
# Basic cleaning
# -----------------------------
df["mp_price"] = pd.to_numeric(df["mp_price"], errors="coerce")
df["mp_month"] = pd.to_numeric(df["mp_month"], errors="coerce")
df["mp_year"] = pd.to_numeric(df["mp_year"], errors="coerce")

df = df.dropna(subset=["adm0_name", "cm_name", "mp_price", "mp_month", "mp_year"])

# Build a monthly date - using a different approach to avoid numpy.bool8
df["date"] = pd.to_datetime({
    'year': df["mp_year"].astype(int),
    'month': df["mp_month"].astype(int),
    'day': np.ones(len(df), dtype=int)
}, errors="coerce")

# -----------------------------
# Focus on Sahel countries
# -----------------------------
sahel_countries = ["Burkina Faso", "Mali", "Niger", "Chad", "Mauritania", "Nigeria"]
# Use == comparison with .isin() instead of direct boolean operations
df_sahel = df[df["adm0_name"].isin(sahel_countries)].copy()

# -----------------------------
# Keep only retail series for most visuals
# -----------------------------
# Use explicit boolean conversion to avoid numpy.bool8
df_retail = df_sahel[df_sahel["pt_name"].str.contains("Retail", case=False, na=False) == True].copy()

# -----------------------------
# Create a simplified commodity label
# Example: "Millet - Retail" -> "Millet"
# -----------------------------
df_retail["commodity"] = df_retail["cm_name"].str.replace(r"\s*-\s*Retail$", "", regex=True)
df_retail["commodity"] = df_retail["commodity"].str.replace(r"\s*-\s*Wholesale$", "", regex=True)

# Month names for seasonal charts
month_order = [1,2,3,4,5,6,7,8,9,10,11,12]
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
month_map = dict(zip(month_order, month_labels))
df_retail["month_label"] = df_retail["mp_month"].map(month_map)

# -----------------------------
# Optional: keep only KG prices for better comparability
# -----------------------------
# Use explicit comparison to avoid numpy.bool8
df_kg = df_retail[df_retail["um_name"] == "KG"].copy()

print(df_kg.shape)
print(df_kg["commodity"].value_counts().head(15))

(158107, 21)
commodity
Millet                    35164
Rice (imported)           20285
Sorghum                   19401
Maize                     17535
Beans (niebe)             17188
Rice (local)              13774
Maize (white)              6954
Sorghum (white)            5843
Groundnuts (unshelled)     5658
Sorghum (red)              2685
Sugar                      1602
Wheat                      1317
Meat (beef)                1162
Sorghum (taghalit)         1120
Groundnuts (shelled)        982
Name: count, dtype: int64


In [5]:
pip show plotly

Name: plotly
Version: 6.6.0
Summary: An open-source interactive data visualization library for Python
Home-page: 
Author: 
Author-email: Chris P <chris@plot.ly>
License: MIT License

Copyright (c) 2016-2024 Plotly Technologies Inc.

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN 

In [6]:
!pip install plotly

In [7]:
pip install --upgrade plotly

  Obtaining dependency information for plotly from https://files.pythonhosted.org/packages/52/d2/c6e44dba74f17c6216ce1b56044a9b93a929f1c2d5bdaff892512b260f5e/plotly-6.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=1.15.1 from https://files.pythonhosted.org/packages/3f/c3/06490e98393dcb4d6ce2bf331a39335375c300afaef526897881fbeae6ab/narwhals-2.18.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 4.0 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.0/445.0 kB 64.2 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: plotly
    Found existing installation: plotly 5.9.0
    Uninstalling plotly-5.9.0:
      Successfully uninstalled plotly-5.9.0
Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
print(np.__version__)

1.26.4


In [12]:
pip install --upgrade numpy

  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/b5/7c/c061f3de0630941073d2598dc271ac2f6cbcf5c83c74a5870fea07488333/numpy-2.4.3-cp311-cp311-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 3.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 2.4.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [14]:
!pip3 install --upgrade numpy

In [15]:
import numpy as np
print(np.__version__)

2.3.5


In [7]:
pip install numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Check units
df_sahel["um_name"].value_counts().head(20)

# Check price types
df_sahel["pt_name"].value_counts()

# Check missingness
df_sahel[["adm0_name", "cm_name", "um_name", "mp_price"]].isna().sum()

# Check exact commodity labels
sorted(df_sahel["cm_name"].dropna().unique())[:100]

['Bananas - Retail',
 'Beans (niebe) - Retail',
 'Beans (niebe) - Wholesale',
 'Beans (red) - Retail',
 'Beans (white) - Retail',
 'Bread - Retail',
 'Cassava meal (gari, yellow) - Retail',
 'Cassava meal (gari, yellow) - Wholesale',
 'Couscous - Retail',
 'Cowpeas (brown) - Retail',
 'Cowpeas (brown) - Wholesale',
 'Cowpeas (white) - Retail',
 'Cowpeas (white) - Wholesale',
 'Cowpeas - Retail',
 'Eggs - Retail',
 'Exchange rate - Retail',
 'Feed (flour) - Retail',
 'Feed (rakhel) - Retail',
 'Feed (wheat bran) - Retail',
 'Fish (smoked) - Retail',
 'Fish - Retail',
 'Fonio - Retail',
 'Fuel (diesel) - Retail',
 'Fuel (petrol-gasoline) - Retail',
 'Gari (white) - Retail',
 'Gari (white) - Wholesale',
 'Groundnuts (shelled) - Retail',
 'Groundnuts (shelled) - Wholesale',
 'Groundnuts (unshelled) - Retail',
 'Groundnuts - Retail',
 'Livestock (camel) - Retail',
 'Livestock (cattle) - Retail',
 'Livestock (goat, medium-sized castrated male) - Retail',
 'Livestock (goat, medium-sized male)

In [9]:
import plotly.express as px
print("Plotly Express is installed ✅")

Plotly Express is installed ✅


In [10]:
import plotly
print(plotly.__version__)

6.6.0


df = px.data.gapminder()
fig = px.line(df, x="year", y="lifeExp", color="country")

fig.show()

In [11]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [12]:
import plotly.express as px

df = px.data.gapminder().query("country == 'Canada'")
fig = px.line(df, x="year", y="lifeExp", title="Test Plot")
fig.show()

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Ensure charts render in Jupyter
pio.renderers.default = "notebook_connected"

# Display options
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

In [14]:
# =========================================
# 2. LOAD DATA
# =========================================

file_path = "wfp_food_prices_database_wacaro.csv"   # adjust if needed
df = pd.read_csv(file_path, low_memory=False)

print("Rows, columns:", df.shape)
df.head()

Rows, columns: (332829, 18)


,adm0_id,adm0_name,adm1_id,adm1_name,mkt_id,mkt_name,cm_id,cm_name,cur_id,cur_name,pt_id,pt_name,um_id,um_name,mp_month,mp_year,mp_price,mp_commoditysource
0,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,1,2002,145.00,NaN
1,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,1,2003,106.00,NaN
2,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,2,2003,107.50,NaN
3,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,3,2003,95.00,NaN
4,29,Benin,609,Alibori,1044,Malanville (CBM),51,Maize - Wholesale,0,XOF,14,Wholesale,5,KG,4,2003,95.00,NaN


In [15]:
# =========================================
# 3. INITIAL DATA REVIEW
# =========================================

print(df.columns.tolist())
print("\nCountries:", df["adm0_name"].nunique())
print("Markets:", df["mkt_name"].nunique() if "mkt_name" in df.columns else "No market column found")
print("Commodities:", df["cm_name"].nunique())
print("Units:", df["um_name"].nunique())
print("Price types:", df["pt_name"].nunique())

df[["adm0_name", "cm_name", "pt_name", "um_name", "mp_price", "mp_year", "mp_month"]].head()

['adm0_id', 'adm0_name', 'adm1_id', 'adm1_name', 'mkt_id', 'mkt_name', 'cm_id', 'cm_name', 'cur_id', 'cur_name', 'pt_id', 'pt_name', 'um_id', 'um_name', 'mp_month', 'mp_year', 'mp_price', 'mp_commoditysource']

Countries: 16
Markets: 672
Commodities: 255
Units: 50
Price types: 2


,adm0_name,cm_name,pt_name,um_name,mp_price,mp_year,mp_month
0,Benin,Maize - Wholesale,Wholesale,KG,145.00,2002,1
1,Benin,Maize - Wholesale,Wholesale,KG,106.00,2003,1
2,Benin,Maize - Wholesale,Wholesale,KG,107.50,2003,2
3,Benin,Maize - Wholesale,Wholesale,KG,95.00,2003,3
4,Benin,Maize - Wholesale,Wholesale,KG,95.00,2003,4


In [18]:
# =========================================
# 4. DATA CLEANING
# =========================================

# Numeric conversion
df["mp_price"] = pd.to_numeric(df["mp_price"], errors="coerce")
df["mp_year"] = pd.to_numeric(df["mp_year"], errors="coerce")
df["mp_month"] = pd.to_numeric(df["mp_month"], errors="coerce")

# Drop rows with critical missing fields
df = df.dropna(subset=["adm0_name", "cm_name", "mp_price", "mp_year", "mp_month"]).copy()

# Create date field
df["date"] = pd.to_datetime(
    dict(
        year=df["mp_year"].astype(int),
        month=df["mp_month"].astype(int),
        day=1
    ),
    errors="coerce"
)

# Standardize commodity label
df["commodity"] = df["cm_name"].astype(str)
df["commodity"] = df["commodity"].str.replace(r"\s*-\s*Retail$", "", regex=True)
df["commodity"] = df["commodity"].str.replace(r"\s*-\s*Wholesale$", "", regex=True)

# Month label
month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}
df["month_label"] = df["mp_month"].map(month_map)

print("Cleaned shape:", df.shape)

Cleaned shape: (332829, 21)


In [19]:
# =========================================
# 5. FILTER TO SAHEL COUNTRIES
# =========================================

sahel_countries = ["Burkina Faso", "Mali", "Niger", "Chad", "Mauritania", "Nigeria"]

df_sahel = df[df["adm0_name"].isin(sahel_countries)].copy()

print("Sahel dataset shape:", df_sahel.shape)
print(df_sahel["adm0_name"].value_counts())

Sahel dataset shape: (208710, 21)
adm0_name
Mali            58173
Niger           51946
Nigeria         41990
Burkina Faso    30948
Chad            16070
Mauritania       9583
Name: count, dtype: int64


In [20]:
# =========================================
# 6. DEFINE ANALYTICAL SUBSETS
# =========================================

# Retail prices only
df_retail = df_sahel[df_sahel["pt_name"].astype(str).str.contains("Retail", case=False, na=False)].copy()

# KG only for price comparability
df_kg = df_retail[df_retail["um_name"] == "KG"].copy()

print("Retail only:", df_retail.shape)
print("Retail + KG only:", df_kg.shape)

print("\nTop commodities:")
print(df_kg["commodity"].value_counts().head(20))

Retail only: (186143, 21)
Retail + KG only: (158107, 21)

Top commodities:
commodity
Millet                    35164
Rice (imported)           20285
Sorghum                   19401
Maize                     17535
Beans (niebe)             17188
Rice (local)              13774
Maize (white)              6954
Sorghum (white)            5843
Groundnuts (unshelled)     5658
Sorghum (red)              2685
Sugar                      1602
Wheat                      1317
Meat (beef)                1162
Sorghum (taghalit)         1120
Groundnuts (shelled)        982
Meat (goat)                 980
Rice (paddy)                760
Peanut                      627
Yam                         623
Sesame                      511
Name: count, dtype: int64


In [14]:
## Analytical note
# To ensure valid comparisons, most charts below are based on:
# - retail prices only
# - kilogram units only
# - selected staple commodities

# This avoids misleading comparisons across different units or price types.

In [22]:
# =========================================
# 7. HELPER FUNCTIONS
# =========================================

def median_price_by_group(data, group_cols):
    """
    Returns median price aggregated by selected group columns.
    """
    return data.groupby(group_cols, as_index=False)["mp_price"].median()


def seasonal_index(data, group_cols, value_col="mp_price"):
    """
    Creates a seasonal index where average = 100 within each group.
    """
    out = data.copy()
    out["seasonal_index"] = (
        out.groupby(group_cols)[value_col]
        .transform(lambda x: 100 * x / x.mean())
    )
    return out


def latest_available_by_country(data, value_col):
    """
    Keeps latest available observation for each country.
    """
    return (
        data.sort_values("date")
        .groupby("adm0_name", as_index=False)
        .tail(1)
        .copy()
    )

In [17]:
## Visual 1. Top staple commodities
# This chart identifies the commodities that dominate the Sahel dataset and helps narrow the analysis to the most relevant staple foods.

In [23]:
# =========================================
# 8. VISUAL 1 — TOP STAPLE COMMODITIES
# =========================================

top_commodities = (
    df_kg["commodity"]
    .value_counts()
    .head(10)
    .reset_index()
)
top_commodities.columns = ["commodity", "observations"]

fig = px.bar(
    top_commodities,
    x="observations",
    y="commodity",
    orientation="h",
    title="Top 10 Staple Commodities by Number of Observations in the Sahel",
    text="observations"
)
fig.update_layout(yaxis=dict(categoryorder="total ascending"))
fig.show()

In [ ]:
## Visual 2. Price trends over time
# This chart compares median staple prices across Sahel countries over time in order to identify structural trends and periods of price stress.

In [24]:
# =========================================
# 9. VISUAL 2 — PRICE TRENDS OVER TIME
# =========================================

selected_commodities = ["Millet", "Sorghum", "Maize", "Rice (imported)"]

trend_df = df_kg[df_kg["commodity"].isin(selected_commodities)].copy()

trend_agg = median_price_by_group(
    trend_df,
    ["date", "adm0_name", "commodity"]
)

fig = px.line(
    trend_agg,
    x="date",
    y="mp_price",
    color="adm0_name",
    facet_row="commodity",
    title="Staple Food Price Trends Across Sahel Countries",
    labels={
        "date": "Date",
        "mp_price": "Median price",
        "adm0_name": "Country"
    }
)
fig.update_yaxes(matches=None)
fig.show()

In [25]:
## Visual 3. Seasonal price patterns
# This visual highlights how staple prices typically evolve across the calendar year and helps identify lean-season pressure.

In [26]:
# =========================================
# 10. VISUAL 3 — SEASONAL PRICE PATTERN
# =========================================

seasonal_commodities = ["Millet", "Sorghum", "Maize"]

season_df = df_kg[df_kg["commodity"].isin(seasonal_commodities)].copy()

season_agg = median_price_by_group(
    season_df,
    ["mp_month", "month_label", "adm0_name", "commodity"]
)

season_agg = seasonal_index(season_agg, ["adm0_name", "commodity"], value_col="mp_price")

fig = px.line(
    season_agg,
    x="mp_month",
    y="seasonal_index",
    color="adm0_name",
    facet_row="commodity",
    markers=True,
    title="Seasonal Price Patterns of Key Staples in the Sahel",
    labels={
        "mp_month": "Month",
        "seasonal_index": "Seasonal Price Index (Average=100)",
        "adm0_name": "Country"
    }
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(month_map.keys()),
    ticktext=list(month_map.values())
)
fig.update_yaxes(matches=None)
fig.show()

In [ ]:
## Visual 4. Imported vs local staple prices
# This chart compares imported rice with key local staples in order to show structural differences in cost and exposure to import dependence.

In [27]:
# =========================================
# 11. VISUAL 4 — IMPORTED VS LOCAL STAPLES
# =========================================

comparison_commodities = ["Rice (imported)", "Millet", "Sorghum", "Maize"]

comp_df = df_kg[df_kg["commodity"].isin(comparison_commodities)].copy()

comp_agg = median_price_by_group(
    comp_df,
    ["adm0_name", "commodity"]
)

fig = px.bar(
    comp_agg,
    x="adm0_name",
    y="mp_price",
    color="commodity",
    barmode="group",
    title="Imported Rice Compared with Local Staples in Sahel Countries",
    labels={
        "adm0_name": "Country",
        "mp_price": "Median price",
        "commodity": "Commodity"
    }
)
fig.show()

In [28]:
## Visual 5. Market dispersion within countries
# This chart shows the range of market-level prices for the same commodity, illustrating how national averages can conceal local disparities.

In [29]:
# =========================================
# 12. VISUAL 5 — MARKET DISPERSION
# =========================================

commodity_name = "Millet"

disp_df = df_kg[df_kg["commodity"] == commodity_name].copy()

market_disp = (
    disp_df.groupby(["adm0_name", "mkt_name"], as_index=False)["mp_price"]
    .median()
)

fig = px.box(
    market_disp,
    x="adm0_name",
    y="mp_price",
    color="adm0_name",
    title=f"Distribution of Market-Level Median Prices for {commodity_name}",
    labels={
        "adm0_name": "Country",
        "mp_price": "Median price"
    }
)
fig.show()

In [ ]:
## Visual 6. Price volatility
# This chart measures month-to-month changes in staple prices and helps identify contexts where market instability itself is a risk factor.

In [30]:
# =========================================
# 13. VISUAL 6 — PRICE VOLATILITY HEATMAP
# =========================================

commodity_name = "Millet"

vol_df = df_kg[df_kg["commodity"] == commodity_name].copy()

vol_agg = (
    vol_df.groupby(["date", "adm0_name"], as_index=False)["mp_price"]
    .median()
    .sort_values(["adm0_name", "date"])
)

vol_agg["pct_change"] = (
    vol_agg.groupby("adm0_name")["mp_price"]
    .pct_change() * 100
)

vol_agg["year_month"] = vol_agg["date"].dt.strftime("%Y-%m")

heatmap_df = vol_agg.pivot(index="adm0_name", columns="year_month", values="pct_change")

fig = go.Figure(
    data=go.Heatmap(
        z=heatmap_df.values,
        x=heatmap_df.columns,
        y=heatmap_df.index,
        colorscale="RdBu",
        zmid=0,
        colorbar_title="% monthly change"
    )
)

fig.update_layout(
    title=f"Monthly Price Volatility Heatmap for {commodity_name}",
    xaxis_title="Month",
    yaxis_title="Country"
)

fig.show()

In [ ]:
## Visual 7. Household purchasing power
# This visual links staple food prices to wage data in order to estimate how many kilograms of food can be purchased with one day of labor.

In [32]:
# =========================================
# 14. VISUAL 7 — PURCHASING POWER
# =========================================

food_commodity = "Millet"
wage_series = "Wage (non-qualified labour, agricultural) - Retail"
country_name = "Niger"

food_df = df_sahel[
    (df_sahel["cm_name"] == f"{food_commodity} - Retail") &
    (df_sahel["um_name"] == "KG")
].copy()

wage_df = df_sahel[
    df_sahel["cm_name"] == wage_series
].copy()

food_agg = (
    food_df.groupby(["date", "adm0_name"], as_index=False)["mp_price"]
    .median()
    .rename(columns={"mp_price": "food_price"})
)

wage_agg = (
    wage_df.groupby(["date", "adm0_name"], as_index=False)["mp_price"]
    .median()
    .rename(columns={"mp_price": "daily_wage"})
)

tot_df = pd.merge(food_agg, wage_agg, on=["date", "adm0_name"], how="inner")
tot_df["kg_affordable"] = tot_df["daily_wage"] / tot_df["food_price"]

plot_df = tot_df[tot_df["adm0_name"] == country_name].copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df["date"],
    y=plot_df["food_price"],
    name=f"{food_commodity} price",
    mode="lines"
))

fig.add_trace(go.Scatter(
    x=plot_df["date"],
    y=plot_df["kg_affordable"],
    name="KG affordable per daily wage",
    mode="lines",
    yaxis="y2"
))

fig.update_layout(
    title=f"{country_name}: {food_commodity} Price and Purchasing Power",
    xaxis=dict(title="Date"),
    yaxis=dict(title="Food price"),
    yaxis2=dict(
        title="KG affordable per daily wage",
        overlaying="y",
        side="right"
    ),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

In [33]:
## Optional visual. Highest-priced markets
# Because geographic coordinates are not included in the dataset, this chart is used instead of a market map to identify hotspot markets.

In [34]:
# =========================================
# 15. OPTIONAL — HOTSPOT MARKET RANKING
# =========================================

commodity_name = "Millet"
year_filter = 2020

hotspot_df = df_kg[
    (df_kg["commodity"] == commodity_name) &
    (df_kg["mp_year"] == year_filter)
].copy()

hotspot_agg = (
    hotspot_df.groupby(["adm0_name", "adm1_name", "mkt_name"], as_index=False)["mp_price"]
    .median()
    .sort_values("mp_price", ascending=False)
    .head(20)
)

hotspot_agg["location"] = (
    hotspot_agg["adm0_name"].astype(str) + " | " +
    hotspot_agg["adm1_name"].fillna("Unknown").astype(str) + " | " +
    hotspot_agg["mkt_name"].astype(str)
)

fig = px.bar(
    hotspot_agg,
    x="mp_price",
    y="location",
    orientation="h",
    color="adm0_name",
    title=f"Top 20 Highest-Priced Markets for {commodity_name} ({year_filter})",
    labels={
        "mp_price": "Median price",
        "location": "Market"
    }
)

fig.update_layout(yaxis=dict(categoryorder="total ascending"))
fig.show()

In [35]:
# Key Analytical Takeaways

## 1. Staple foods drive the Sahel market story
#A limited number of staples such as millet, sorghum, maize, and imported rice dominate the price data and should remain central to market monitoring.

## 2. Price stress is seasonal
#Seasonal charts show recurring increases during the lean season, supporting early warning and response timing.

## 3. Imported staples are structurally more expensive
#Imported rice tends to cost more than local staples, suggesting higher vulnerability in import-dependent areas.

## 4. Market fragmentation matters
#Price dispersion across markets indicates that national averages may hide localized stress.

## 5. Affordability is critical
#Purchasing power analysis provides a more direct measure of household vulnerability than prices alone.

In [37]:
# =========================================
# 16. OPTIONAL EXPORTS
# =========================================

# Example: save last figure as html
# fig.write_html("purchasing_power_niger.html")

# Example: export aggregated tables
trend_agg.to_csv("trend_agg_sahel.csv", index=False)
season_agg.to_csv("seasonal_agg_sahel.csv", index=False)
comp_agg.to_csv("imported_vs_local_sahel.csv", index=False)
market_disp.to_csv("market_dispersion_millet.csv", index=False)
tot_df.to_csv("purchasing_power_sahel.csv", index=False)

print("Exports completed.")

Exports completed.


In [38]:
#1. Title and objective
#2. Setup and imports
#3. Load data
#4. Initial data review
#5. Data cleaning
#6. Filter to Sahel scope
#7. Define analytical subsets
#8. Analytical note
#9. Helper functions
#10. Visual 1 — Top staple commodities
#11. Visual 2 — Price trends over time
#12. Visual 3 — Seasonal price patterns
#13. Visual 4 — Imported vs local staple comparison
#14. Visual 5 — Market dispersion
#15. Visual 6 — Volatility heatmap
#16. Visual 7 — Purchasing power
#17. Optional hotspot chart
#18. Key findings summary
#19. Optional export